In [337]:
import random
discount = 0.5

def step_left(state):
  if(state!=1):
    return state-1
  else:
    return -1

def step_right(state):
  if(state!=6):
    return state+1
  else:
    return -1

def step_null(state):
  return state

# print("\n\n")
# for i in range(1,7):
#   print(f" {step_left(i)} <- state {i} ")
#   print(f"state {i} -> {step_right(i)} ")
# print("\n\n")

def new_state(state,action):
  if(action == "L"):
    return step_left(state)
  elif(action == "R"):
    return step_right(state)
  else:
    return step_null(state)

def reward(state):
  if(state==1):
    return 100
  elif(state==6):
    return 40
  else:
    return 0

#all right policy, absolute, returns probability for L and R
def policy_r(s,a):
  if(s!=1 and s!=6):
    if(a=="R"):
      return 1
    else:
      return 0
  elif(a=="0"):
    return 1
  else:
    return 0

def policy_l(s,a):
  if(s!=1 and s!=6):
    if(a=="R" or a=="0"):
      return 0
    else:
      return 1
  elif(a=="0"):
    return 1
  else:
    return 0

def policy_mixed(s,a):
  if(s==6 or s==1):
    return 1 if(a=="0") else 0
  elif(s==5):
    return 1 if(a=="R") else 0
  else:
    return 1 if(a=="L") else 0

def policy_weighted(s,a):
  if(s==6 or s==1):
    return 1 if(a=="0") else 0
  else:
    return 0.6 if (a=="L") else 0.4 if(a=="R") else 0.0


def policy_random(s,a):
  if(s==6 or s==1):
    return 1 if(a=="0") else 0
  else:
    return 1/2 if (a=="L") else 1/2 if(a=="R") else 0

def policy_random2(s,a):
  if(s==6 or s==1):
    return 1 if(a=="0") else 0
  else:
    return 1/3 if (a=="L") else 1/3 if(a=="R") else 1/3


def modified_policy(s,a,rets):
  Q1 = rets.get((s,"L"),0)
  Q2 = rets.get((s,"0"),0)
  Q3 = rets.get((s,"R"),0)
  sum = Q1 + Q2 + Q3
  if sum==0:
    print("sum is zero")
    return 1/3
  pa_s = (Q1 if(a=="L") else Q2 if(a=="0") else Q3)/sum
  return pa_s

def policy_dist_default(s):

  return [policy_random2(s,"L"),policy_random2(s,"0"),policy_random2(s,"R")]

def policy_dist(s,rets=None):
  if rets is None:
    return policy_dist_default(s)
  return [modified_policy(s,"L",rets),modified_policy(s,"0",rets),modified_policy(s,"R",rets)]



def calculate_return(trajectory,discount=0.5):

  for i in range(len(trajectory) - 1, -1, -1):
    if(trajectory[i][0]==6 or trajectory[i][0]==1):
      trajectory[i][2] = reward(trajectory[i][0])
      trajectory[i][3] = trajectory[i][2]
    else:
      trajectory[i][3] = reward(trajectory[i][0]) + discount * trajectory[i+1][3]

  return trajectory

  #returned trajectory looks like [s,"action",reward,return]



def sampler(state,returns=None):
  trajectory = []
  s_init = state
  count = 0
  

  while(s_init!=1 and s_init!=6):

    if returns is None:
      action = random.choices(["L","0","R"],weights=policy_dist(s_init),k=1)[0]
    
    else:
      action = random.choices(["L","0","R"],weights=policy_dist(s_init,returns),k=1)[0]

    #print(f"current state: {s_init} action:{action} new state: {new_state(s_init,action)} count: {count} policy dist: {policy_dist(s_init)}")
    #print(f"{[s_init,action]}")
    trajectory.append([s_init,action,0,0])
    s_init = new_state(s_init,action)
    count = count + 1
    if((s_init!=6 or s_init!=1) and count>20000):
      return None
      break
  trajectory.append([s_init,random.choices(["L","0","R"],weights=policy_dist(s_init),k=1)[0],0,0])
  return trajectory




In [338]:



print("state:--- policy distribution:  L  0  R")
for i in range(1,7):
  print(f" {i}        policy distribution:  {policy_dist(i)[0]}  {policy_dist(i)[1]}  {policy_dist(i)[2]}")


trajectory = sampler(4)

trajectory = calculate_return(trajectory,discount=0.995)
print("state   action   reward   return")
for i in trajectory:
  print(f"{i[0]}       {i [1]}        {i [2]}        {i [3]}")



state:--- policy distribution:  L  0  R
 1        policy distribution:  0  1  0
 2        policy distribution:  0.3333333333333333  0.3333333333333333  0.3333333333333333
 3        policy distribution:  0.3333333333333333  0.3333333333333333  0.3333333333333333
 4        policy distribution:  0.3333333333333333  0.3333333333333333  0.3333333333333333
 5        policy distribution:  0.3333333333333333  0.3333333333333333  0.3333333333333333
 6        policy distribution:  0  1  0
state   action   reward   return
4       R        0        38.81490037425062
5       L        0        39.009950124875
4       L        0        39.205980024999995
3       R        0        39.402995
4       R        0        39.601
5       R        0        39.8
6       0        40        40


In [339]:
returns = {}
counts = {}

#finds an average return value for each (s,a) combination
def return_estimation(start_state, num_episodes,rets=None,discount=0.5):
    returns = {}
    counts = {}

    for i in range(num_episodes):
      if rets is None:
        #("we are here")
        trajectory = sampler(start_state)
        #(trajectory)
      else:
        trajectory = sampler(start_state,rets)
        
      if trajectory is not None:
        #print(trajectory)
        trajectory = calculate_return(trajectory,discount=discount)
        for state, action, r, Gt in trajectory:
            key = (state, action)
            if key in returns:
                counts[key] += 1
                # update running average
                returns[key] += (Gt - returns[key]) / counts[key]
            else:
                returns[key] = Gt
                counts[key] = 1

    return returns


# count = 0
# for i in range(2000):
#    returns = return_estimation(4, 40)
#    if(len(returns)!=14):
#       count=count+1

# print(count)

returns = return_estimation(4, 40,discount=0.995)

print(len(returns))
# Print results
print("state   action   average_return")
for (state, action), avg_return in sorted(returns.items()):
    print(f"{state:<7}{action:<8}{avg_return:.6f}")


14
state   action   average_return
1      0       100.000000
2      0       83.104205
2      L       99.500000
2      R       63.226718
3      0       74.858628
3      L       82.048801
3      R       61.574983
4      0       66.509056
4      L       74.354580
4      R       47.582291
5      0       43.999250
5      L       60.171330
5      R       39.800000
6      0       40.000000


In [340]:



def eval(s=4,rets=None,discount=0.5):
  if rets is None:
    returns = return_estimation(4, 10,discount=discount)
  else:
    returns=rets

  trajectory = sampler(s,returns)
  trajectory = calculate_return(trajectory,discount=discount)
  # print("state   action   reward   return")
  # for i in trajectory:
  #   print(f"{i[0]}       {i [1]}        {i [2]}        {i [3]}")
  #print("Trajectory")
  states = [row[0] for row in trajectory]
  #print(states)
  return states



print("state:--- policy distribution:  L  0  R")
for i in range(1,7):
  print(f" {i}        policy distribution:  {policy_dist(i,returns)[0]}  {policy_dist(i,returns)[1]}  {policy_dist(i,returns)[2]}")





state:--- policy distribution:  L  0  R
 1        policy distribution:  0.0  1.0  0.0
 2        policy distribution:  0.40474973104411593  0.3380543168828307  0.25719595207305335
 3        policy distribution:  0.37553961442083234  0.34262999491487584  0.28183039066429183
 4        policy distribution:  0.3945671891396836  0.35293443152462456  0.252498379335692
 5        policy distribution:  0.4179418443239252  0.30561278740612874  0.2764453682699461
 6        policy distribution:  0.0  1.0  0.0


In [ ]:


# First we find expected return for (s,a) using the default or equalized policy distribution
# Then we PASS that expected return list to our policy function to redistribute the probability based on that return list
# This gives us a NEW return list, where actions that previously gained more return now gains even MORE return 
# This is again passed in the return calculator that again passes it to the policy function to redistribute based on this return
# This again makes actions with more return aggregate more
# This continues untill iteration ends



def naive_RL(iterations=100,discount=0.5):
  print("state:--- policy distribution:  L  0  R BEFORE ITERATION")
  for i in range(1,7):
    print(f" {i}        policy distribution:  {policy_dist(i)[0]}  {policy_dist(i)[1]}  {policy_dist(i)[2]}")

  #convergence achieved at 100000
  returns = return_estimation(4, 10,discount=discount)
  
  for i in range(iterations):
    
    returns = return_estimation(4, 50,returns,discount=discount)

  
  print(f"state:--- policy distribution:  L  0  R after all Iterations ")
  for i in range(1,7):
    print(f" {i}        policy distribution:  {policy_dist(i,returns)[0]}  {policy_dist(i,returns)[1]}  {policy_dist(i,returns)[2]}")

  return returns



returns = naive_RL(50000,discount=0.995)

#THIS IS OUR OPTIMAL POLICY, or rather OUR DISTRIBUTION

print("After dust is settled")
 
for i in range(1,7):
    print(f" {i}        policy distribution:  {policy_dist(i,returns)[0]}  {policy_dist(i,returns)[1]}  {policy_dist(i,returns)[2]}")

  

state:--- policy distribution:  L  0  R BEFORE ITERATION
 1        policy distribution:  0  1  0
 2        policy distribution:  0.3333333333333333  0.3333333333333333  0.3333333333333333
 3        policy distribution:  0.3333333333333333  0.3333333333333333  0.3333333333333333
 4        policy distribution:  0.3333333333333333  0.3333333333333333  0.3333333333333333
 5        policy distribution:  0.3333333333333333  0.3333333333333333  0.3333333333333333
 6        policy distribution:  0  1  0
state:--- policy distribution:  L  0  R after all Iterations 
 1        policy distribution:  0.0  1.0  0.0
 2        policy distribution:  0.3688364011750053  0.32317659852329406  0.30798700030170056
 3        policy distribution:  0.35653779433298777  0.3419114559594273  0.30155074970758483
 4        policy distribution:  0.3896521785753599  0.3406248858853602  0.2697229355392799
 5        policy distribution:  0.4102543133315418  0.3595589520086524  0.2301867346598057
 6        policy distri

In [342]:
track = []
for i in range(100):
  track.append(eval(4,returns))


from collections import Counter
# Convert inner lists to tuples
track_tuples = [tuple(x) for x in track]
# Count occurrences
counts = Counter(track_tuples)
# Total number of elements
total = len(track)
# Calculate percentages
percentages = {key: (value / total) * 100 for key, value in counts.items()}
# Sort percentages by value descending and take top 3
top3 = sorted(percentages.items(), key=lambda x: x[1], reverse=True)[:]
# Print nicely
for element, pct in top3:
    print(f"{list(element)}: {pct:.2f}%")  # convert back to list if needed


[4, 3, 2, 1]: 10.00%
[4, 5, 6]: 7.00%
[4, 3, 3, 2, 1]: 3.00%
[4, 3, 2, 2, 1]: 2.00%
[4, 4, 5, 5, 6]: 2.00%
[4, 5, 5, 4, 3, 4, 3, 2, 3, 4, 3, 4, 4, 5, 6]: 1.00%
[4, 4, 4, 4, 4, 5, 6]: 1.00%
[4, 3, 2, 3, 3, 4, 3, 2, 1]: 1.00%
[4, 4, 5, 4, 4, 3, 4, 5, 5, 4, 5, 4, 3, 3, 3, 4, 3, 2, 3, 3, 4, 4, 5, 5, 4, 3, 3, 4, 3, 4, 3, 3, 4, 3, 2, 1]: 1.00%
[4, 5, 4, 4, 3, 4, 5, 5, 5, 5, 6]: 1.00%
[4, 3, 3, 3, 3, 2, 1]: 1.00%
[4, 3, 3, 3, 2, 1]: 1.00%
[4, 5, 5, 5, 5, 5, 5, 6]: 1.00%
[4, 3, 4, 3, 4, 4, 3, 2, 2, 1]: 1.00%
[4, 3, 4, 4, 3, 2, 3, 2, 2, 3, 3, 2, 2, 1]: 1.00%
[4, 5, 4, 4, 3, 3, 4, 5, 5, 5, 5, 5, 4, 3, 3, 2, 1]: 1.00%
[4, 4, 3, 2, 3, 4, 5, 5, 5, 4, 4, 3, 3, 4, 5, 4, 5, 4, 5, 6]: 1.00%
[4, 3, 3, 4, 4, 4, 5, 4, 4, 4, 4, 3, 3, 2, 1]: 1.00%
[4, 3, 2, 2, 3, 4, 3, 3, 3, 4, 3, 3, 4, 5, 5, 4, 4, 5, 5, 6]: 1.00%
[4, 4, 4, 3, 3, 2, 1]: 1.00%
[4, 4, 4, 3, 4, 4, 4, 4, 5, 6]: 1.00%
[4, 4, 5, 4, 5, 4, 4, 5, 5, 4, 4, 5, 4, 5, 4, 3, 2, 1]: 1.00%
[4, 4, 3, 2, 1]: 1.00%
[4, 4, 5, 5, 4, 5, 4, 5, 6]: 1.00%
[4, 3, 3,